In [0]:
# Databricks notebook source
# ============================================================
# OpsIntel Copilot
# Week 6 — Correlation Engine
#
# Purpose:
# Links pipeline failures with suspicious security events
# using time-window rules and generates confidence-scored
# correlation records
#
# Correlations detected:
# 1. Pipeline failure + config change within 30 min
# 2. Privilege escalation + API token rotation within 10 min
# 3. Large data export + impossible travel within 2 hours
# 4. Brute force + pipeline failure within 1 hour
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA  = "opsintel_copilot"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ── Load tables ───────────────────────────────────────────────
pipeline_logs   = spark.table(f"{CATALOG}.{SCHEMA}.bronze_pipeline_logs")
security_alerts = spark.table(f"{CATALOG}.{SCHEMA}.security_alerts_detected")
admin_events    = spark.table(f"{CATALOG}.{SCHEMA}.silver_admin_events")
security_logs   = spark.table(f"{CATALOG}.{SCHEMA}.silver_security_logs")

# Filter to failed pipelines only
failed_pipelines = (
    pipeline_logs
    .filter(F.col("status") == "failed")
    .select(
        "run_id",
        "pipeline_name",
        "region",
        "start_time",
        "end_time",
        "error_message"
    )
    .withColumn("start_time", F.col("start_time").cast("timestamp"))
    .withColumn("end_time", F.col("end_time").cast("timestamp"))
)

print(f"Failed pipeline runs: {failed_pipelines.count()}")
print(f"Security alerts: {security_alerts.count()}")
print(f"Admin events: {admin_events.count()}")

# COMMAND ----------
# ── Correlation 1: Pipeline failure + config change ───────────
# IF config change happened within 30 minutes BEFORE pipeline failure
# THEN high confidence this caused the failure

config_changes = (
    admin_events
    .filter(
        (F.col("action") == "config_changed") |
        (F.col("is_config_change") == True)
    )
    .select(
        "admin_event_id",
        "admin_user",
        "action",
        "target_service",
        "region",
        "event_time"
    )
)

corr1 = (
    failed_pipelines.alias("p")
    .join(
        config_changes.alias("c"),
        on=(
            F.col("p.pipeline_name").contains(
                F.regexp_extract(F.col("c.target_service"), r"(\w+)-pipeline", 0)
            ) &
            (
                F.unix_timestamp(F.col("p.start_time")) -
                F.unix_timestamp(F.col("c.event_time"))
            ).between(0, 1800)
        ),
        how="inner"
    )
    .select(
        F.concat(F.lit("CORR1_"), F.col("p.run_id"), F.lit("_"), F.col("c.admin_event_id"))
            .alias("correlation_id"),
        F.lit("security_to_pipeline").alias("correlation_type"),
        F.col("p.run_id").alias("pipeline_run_id"),
        F.col("p.pipeline_name"),
        F.col("p.start_time").alias("pipeline_failed_at"),
        F.col("p.error_message"),
        F.col("c.admin_event_id").alias("security_event_id"),
        F.col("c.action").alias("security_event_type"),
        F.col("c.event_time").alias("security_event_at"),
        F.col("c.admin_user").alias("involved_user"),
        (
            (F.unix_timestamp(F.col("p.start_time")) -
             F.unix_timestamp(F.col("c.event_time"))) / 60
        ).alias("time_diff_minutes"),
        F.lit(0.90).alias("confidence_score"),
        F.concat(
            F.lit("Config change on "),
            F.col("c.target_service"),
            F.lit(" by "),
            F.col("c.admin_user"),
            F.lit(" occurred before pipeline failure")
        ).alias("recommendation"),
        F.current_timestamp().alias("correlated_at")
    )
)

print(f"Correlation 1 (pipeline + config change): {corr1.count()}")

# COMMAND ----------
# ── Correlation 2: Privilege escalation + token rotation ──────
# IF privilege escalation followed by API token rotation within 10 min
# THEN lateral movement pattern detected

escalations = (
    security_logs
    .filter(F.col("event_type") == "privilege_escalation")
    .select("event_id", "user_email", "event_time", "region")
    .withColumnRenamed("event_id", "esc_event_id")
    .withColumnRenamed("event_time", "esc_time")
    .withColumnRenamed("user_email", "esc_user")
)

token_rotations = (
    security_logs
    .filter(F.col("event_type") == "api_token_rotation")
    .select("event_id", "user_email", "event_time")
    .withColumnRenamed("event_id", "token_event_id")
    .withColumnRenamed("event_time", "token_time")
    .withColumnRenamed("user_email", "token_user")
)

corr2 = (
    escalations.alias("e")
    .join(
        token_rotations.alias("t"),
        on=(
            (F.col("e.esc_user") == F.col("t.token_user")) &
            (
                F.unix_timestamp(F.col("t.token_time")) -
                F.unix_timestamp(F.col("e.esc_time"))
            ).between(0, 600)
        ),
        how="inner"
    )
    .select(
        F.concat(F.lit("CORR2_"), F.col("e.esc_event_id"), F.lit("_"), F.col("t.token_event_id"))
            .alias("correlation_id"),
        F.lit("lateral_movement").alias("correlation_type"),
        F.lit(None).cast("string").alias("pipeline_run_id"),
        F.lit(None).cast("string").alias("pipeline_name"),
        F.lit(None).cast("timestamp").alias("pipeline_failed_at"),
        F.lit(None).cast("string").alias("error_message"),
        F.col("e.esc_event_id").alias("security_event_id"),
        F.lit("privilege_escalation_then_token_rotation").alias("security_event_type"),
        F.col("e.esc_time").alias("security_event_at"),
        F.col("e.esc_user").alias("involved_user"),
        (
            (F.unix_timestamp(F.col("t.token_time")) -
             F.unix_timestamp(F.col("e.esc_time"))) / 60
        ).alias("time_diff_minutes"),
        F.lit(0.88).alias("confidence_score"),
        F.concat(
            F.lit("Lateral movement detected: "),
            F.col("e.esc_user"),
            F.lit(" escalated privileges then rotated API token within 10 minutes")
        ).alias("recommendation"),
        F.current_timestamp().alias("correlated_at")
    )
)

print(f"Correlation 2 (privilege escalation + token rotation): {corr2.count()}")

# COMMAND ----------
# ── Correlation 3: Large export + impossible travel ───────────
# IF large data export followed by impossible travel within 2 hours
# THEN data exfiltration pattern

exports = (
    security_logs
    .filter(F.col("event_type") == "large_data_export")
    .select("event_id", "user_email", "event_time", "region")
    .withColumnRenamed("event_id", "export_event_id")
    .withColumnRenamed("event_time", "export_time")
    .withColumnRenamed("user_email", "export_user")
)

travel = (
    security_logs
    .filter(F.col("event_type") == "impossible_travel")
    .select("event_id", "user_email", "event_time")
    .withColumnRenamed("event_id", "travel_event_id")
    .withColumnRenamed("event_time", "travel_time")
    .withColumnRenamed("user_email", "travel_user")
)

corr3 = (
    exports.alias("ex")
    .join(
        travel.alias("tr"),
        on=(
            (
                F.abs(
                    F.unix_timestamp(F.col("ex.export_time")) -
                    F.unix_timestamp(F.col("tr.travel_time"))
                )
            ) < 7200
        ),
        how="inner"
    )
    .select(
        F.concat(F.lit("CORR3_"), F.col("ex.export_event_id"), F.lit("_"), F.col("tr.travel_event_id"))
            .alias("correlation_id"),
        F.lit("data_exfiltration").alias("correlation_type"),
        F.lit(None).cast("string").alias("pipeline_run_id"),
        F.lit(None).cast("string").alias("pipeline_name"),
        F.lit(None).cast("timestamp").alias("pipeline_failed_at"),
        F.lit(None).cast("string").alias("error_message"),
        F.col("ex.export_event_id").alias("security_event_id"),
        F.lit("large_export_with_impossible_travel").alias("security_event_type"),
        F.col("ex.export_time").alias("security_event_at"),
        F.col("ex.export_user").alias("involved_user"),
        (
            F.abs(
                F.unix_timestamp(F.col("ex.export_time")) -
                F.unix_timestamp(F.col("tr.travel_time"))
            ) / 60
        ).alias("time_diff_minutes"),
        F.lit(0.85).alias("confidence_score"),
        F.concat(
            F.lit("Possible data exfiltration: large export by "),
            F.col("ex.export_user"),
            F.lit(" correlated with impossible travel event")
        ).alias("recommendation"),
        F.current_timestamp().alias("correlated_at")
    )
)

print(f"Correlation 3 (large export + impossible travel): {corr3.count()}")

# COMMAND ----------
# ── Combine all correlations ──────────────────────────────────

all_correlations = (
    corr1
    .union(corr2)
    .union(corr3)
)

total = all_correlations.count()
print(f"\nTotal correlations found: {total}")
display(all_correlations.orderBy("confidence_score", ascending=False))

# COMMAND ----------
# ── Write to Delta table ──────────────────────────────────────

CORRELATIONS_PATH = "s3://opsintel-copilot-angad-0025/gold/correlations/"

(
    all_correlations.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(CORRELATIONS_PATH)
)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.correlation_records")
spark.sql(f"""
    CREATE TABLE {CATALOG}.{SCHEMA}.correlation_records
    USING DELTA
    LOCATION '{CORRELATIONS_PATH}'
""")

print(f"Correlations written to: {CATALOG}.{SCHEMA}.correlation_records")
print(f"Total correlations: {spark.table(f'{CATALOG}.{SCHEMA}.correlation_records').count()}")

# COMMAND ----------
# ── Summary by correlation type ───────────────────────────────

print("\nCorrelation summary by type:")
display(
    spark.table(f"{CATALOG}.{SCHEMA}.correlation_records")
    .groupBy("correlation_type")
    .agg(
        F.count("*").alias("count"),
        F.avg("confidence_score").alias("avg_confidence"),
        F.max("confidence_score").alias("max_confidence")
    )
    .orderBy("count", ascending=False)
)